In [1]:
import sys
sys.path.append("..")           

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from ml.preprocess import (
    add_features, build_preprocessor,
    TARGET, FEATURE_COLUMNS, NUMERIC_FEATURES,
)

pd.set_option("display.max_columns", None)

df = pd.read_csv("../dataset/loan_data_cleaned.csv")
print("Rows, Columns:", df.shape)

Rows, Columns: (32409, 12)


In [2]:
# Pehla check: loan_percent_income = loan / income j che ke?
lti = (df["loan_amnt"] / df["person_income"]).round(2)
print(f"loan_percent_income sathe match: {(lti == df['loan_percent_income']).mean() * 100:.1f} %")

df = add_features(df)
df[["loan_amnt", "person_income", "loan_int_rate", "loan_to_income", "interest_to_income"]].head()

loan_percent_income sathe match: 96.9 %


,loan_amnt,person_income,loan_int_rate,loan_to_income,interest_to_income
0,1000,9600,11.14,0.104167,0.011604
1,5500,9600,12.87,0.572917,0.073734
2,35000,65500,15.23,0.534351,0.081382
3,35000,54400,14.27,0.643382,0.091811
4,2500,9900,7.14,0.252525,0.018030


In [3]:
df = df.drop(columns=["loan_percent_income"])
print("Columns:", list(df.columns))

Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_to_income', 'interest_to_income']


In [4]:
corr = df[NUMERIC_FEATURES + [TARGET]].corr()

print("Target sathe correlation (sauthi vadhare upar):")
print(corr[TARGET].drop(TARGET).sort_values(ascending=False).round(3))

print()
print("Features vache ghani vadhare correlation (> 0.7):")
pairs = corr.drop(index=TARGET, columns=TARGET).abs().where(lambda x: np.triu(np.ones(x.shape), 1).astype(bool)).stack()
print(pairs[pairs > 0.7].round(3))

Target sathe correlation (sauthi vadhare upar):
interest_to_income            0.459
loan_to_income                0.386
loan_int_rate                 0.334
loan_amnt                     0.106
cb_person_cred_hist_length   -0.016
person_age                   -0.022
person_emp_length            -0.086
person_income                -0.169
Name: loan_status, dtype: float64

Features vache ghani vadhare correlation (> 0.7):
person_age      cb_person_cred_hist_length    0.878
loan_to_income  interest_to_income            0.893
dtype: float64


In [5]:
X = df[FEATURE_COLUMNS]
y = df[TARGET]
print("X shape:", X.shape, "| y shape:", y.shape)
print(FEATURE_COLUMNS)

X shape: (32409, 12) | y shape: (32409,)
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'cb_person_cred_hist_length', 'loan_to_income', 'interest_to_income', 'loan_grade', 'cb_person_default_on_file', 'person_home_ownership', 'loan_intent']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Default % in train: {y_train.mean() * 100:.1f}")
print(f"Default % in test : {y_test.mean() * 100:.1f}")

Train: (25927, 12) | Test: (6482, 12)
Default % in train: 21.9
Default % in test : 21.9


In [7]:
preprocessor = build_preprocessor()

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print("Train processed:", X_train_p.shape)
print("Test processed :", X_test_p.shape)
print()
print("Final columns:")
for c in X_train_p.columns:
    print("  ", c)

Train processed: (25927, 20)
Test processed : (6482, 20)

Final columns:
   num__person_age
   num__person_income
   num__person_emp_length
   num__loan_amnt
   num__loan_int_rate
   num__cb_person_cred_hist_length
   num__loan_to_income
   num__interest_to_income
   ord__loan_grade
   ord__cb_person_default_on_file
   nom__person_home_ownership_MORTGAGE
   nom__person_home_ownership_OTHER
   nom__person_home_ownership_OWN
   nom__person_home_ownership_RENT
   nom__loan_intent_DEBTCONSOLIDATION
   nom__loan_intent_EDUCATION
   nom__loan_intent_HOMEIMPROVEMENT
   nom__loan_intent_MEDICAL
   nom__loan_intent_PERSONAL
   nom__loan_intent_VENTURE


In [8]:
X_train_p.head()

,num__person_age,num__person_income,num__person_emp_length,num__loan_amnt,num__loan_int_rate,num__cb_person_cred_hist_length,num__loan_to_income,num__interest_to_income,ord__loan_grade,ord__cb_person_default_on_file,nom__person_home_ownership_MORTGAGE,nom__person_home_ownership_OTHER,nom__person_home_ownership_OWN,nom__person_home_ownership_RENT,nom__loan_intent_DEBTCONSOLIDATION,nom__loan_intent_EDUCATION,nom__loan_intent_HOMEIMPROVEMENT,nom__loan_intent_MEDICAL,nom__loan_intent_PERSONAL,nom__loan_intent_VENTURE
31112,1.510006,0.365413,3.587041,0.066435,-0.324174,2.526875,-0.495240,-0.525605,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5364,-0.436694,-0.300768,0.563333,-0.171904,1.453572,-0.443329,-0.005785,0.521507,4.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
13349,-0.274469,0.460581,-0.948522,1.655355,-0.012835,-0.690846,0.482449,0.362532,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
31695,2.158905,0.436066,-0.696546,1.019787,-0.025289,2.279358,0.091062,0.034953,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
24075,1.185556,-0.510139,-0.696546,0.066435,0.662771,0.051705,0.802078,1.015975,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [9]:
print("Missing in train:", X_train_p.isnull().sum().sum())
print("Missing in test :", X_test_p.isnull().sum().sum())
print()
check = X_train_p[["num__person_age", "num__person_income", "num__loan_int_rate"]]
print("Scaled columns - mean ~0, std ~1:")
print(check.agg(["mean", "std"]).round(3))

Missing in train: 0
Missing in test : 0

Scaled columns - mean ~0, std ~1:
      num__person_age  num__person_income  num__loan_int_rate
mean              0.0                -0.0                 0.0
std               1.0                 1.0                 1.0


In [10]:
train_out = X_train_p.copy()
train_out[TARGET] = y_train.values
test_out = X_test_p.copy()
test_out[TARGET] = y_test.values

train_out.to_csv("../dataset/train_processed.csv", index=False)
test_out.to_csv("../dataset/test_processed.csv", index=False)
joblib.dump(preprocessor, "../ml/saved_models/preprocessor.joblib")

print("Saved: dataset/train_processed.csv", train_out.shape)
print("Saved: dataset/test_processed.csv ", test_out.shape)
print("Saved: ml/saved_models/preprocessor.joblib")

Saved: dataset/train_processed.csv (25927, 21)
Saved: dataset/test_processed.csv  (6482, 21)
Saved: ml/saved_models/preprocessor.joblib


In [11]:
loaded = joblib.load("../ml/saved_models/preprocessor.joblib")

# ek kalpnik applicant (Django form jevu j data)
sample = pd.DataFrame([{
    "person_age": 28, "person_income": 600000, "person_emp_length": 4,
    "loan_amnt": 150000, "loan_int_rate": 11.5, "cb_person_cred_hist_length": 5,
    "loan_grade": "C", "cb_person_default_on_file": "N",
    "person_home_ownership": "RENT", "loan_intent": "EDUCATION",
}])

sample = add_features(sample)[FEATURE_COLUMNS]
print("Processed shape:", loaded.transform(sample).shape)

Processed shape: (1, 20)
